## 턴에 대한 기본 실습
    



In [66]:
# 상태 저장용 전역 변수
state = {
    "turn":0, # 현재턴수
    "history" : []
}

In [67]:
# 턴별 대화 함수 설계
def phase1(user_input):
  # 필요한것은 고객에게 어떤 답변을 받을지 설계하는것이 필요
  # 고객 현재 상태 / 기본 통계정보(나이, 연령 등)/
  # 내용에는 프롬프트 내용들이 들어간다.
  return f"반가워요{user_input}에 대해서 조금 더 알려주세요"
  # return부분에 향후 AI답변이 들어간다.

def phase2(user_input): # user_input 대신에 phase1에 대한 결과물
  return f"{user_input}에 대해서 더 정확하게 알려주세요"



In [68]:
# 턴 분기 함수에 대해서 작성!
# 위에서 작성한 함수가 실제로 구동되는 코드

def run_turn(user_input):
  # phase분기
  # 위에서 정의한 프롬프트 + Langchain의 실행
  if state["turn"] == 0 :
    output = phase1(user_input)
  else:
    output = phase2(user_input)

  # 상태 업데이트
  # 위에 있는 if문의 실행이 될때마다 history에 저장!
  state["history"].append({"user": user_input, "bot": output})
  state['turn'] +=

  # 5턴 마다 요약 이벤트(가정)
  if state['turn'] % 5 == 0:
      print('요약 이벤트 발생')
      print('현재까지 내용 요약', summarize_history()) #langchain에 있는 요약함수!

  return output

SyntaxError: invalid syntax (ipython-input-519905006.py, line 15)

In [ ]:
# 요약 함수 간단하게 추가!
def summarize_history():
  recent = [h['user'] for h in state['history'][-5:]]
  return " / ".join(recent)

In [ ]:
for msg in ["안녕하세요", "오늘 날씨 어때요?", "SQL 공부는 어렵네요", "추천 강의 있나요?", "감사합니다"]:
    print(f"\n👤 사용자: {msg}")
    res = run_turn(msg)
    print(f"🤖 챗봇: {res}")

## 라이브러리 임포트


In [ ]:
# API Key 설정
import os

# os.environ["OPENAI_API_KEY"] = ""  # ← 수강생 개인 키 입력

In [ ]:
!pip install -q langchain langchain-openai

In [76]:
# 라이브러리 임포트 - 기본 LCEL구성을 위한 라이브러리
from langchain_openai import ChatOpenAI # 모델
from langchain.prompts import ChatPromptTemplate # 템플릿
from langchain_core.output_parsers import StrOutputParser # 아웃풋 파서

# 라이브러리 추가 요약 메모리
from langchain.memory import ConversationSummaryBufferMemory

In [77]:
# LLM 모델 초기화
# GPT 모델 설정
llm = ChatOpenAI(model='gpt-4o-mini', temperature = 0.7)


In [78]:
# 최근 내용을 기억 + 요약해주는 ConversationSummaryBufferMemory
# gpt-4o-mini가 여러분들과 챗봇의 대화내용을 요약해서 500토큰 내외로 수행해주는 메모리를 구현!
memory = ConversationSummaryBufferMemory(
    llm = llm,
    max_token_limit=500, # 요약에 대해서 좀더 필요한 토큰 수를 조절!
    memory_key = 'history'
)

In [79]:
# 1단계 : 기본 프롬프트 템플릿을 정의
# 2단계 : 3단계 역할 프롬프트를 설정
# 1턴은 친절한 상담가
# 2턴은 분석하는 상담가
# 3턴은 해결사

prompts = {

1 : ChatPromptTemplate.from_template("""
너는 친절한 상담사야. 고객의 말을 따뜻하게 공감하고 짧은 질문으로 대화를 이어가는 챗봇이야.
이전 대화 : {history}
고객 : {input}
상담사 :
"""),
2 : ChatPromptTemplate.from_template("""
너는 분석하는 상담가야. 구체적인 상황, 원인, 빈도를 물어봐봐.
이전 대화 : {history}
고객 : {input}
상담사 :
"""),
3 : ChatPromptTemplate.from_template("""
너는 해결사야. 지금까지의 대화를 요약하고, 실행가능한 해결책 3가지를 제시해줘
이전 대화 : {history}
고객 : {input}
상담사 :
""")
}

In [80]:
prompts[4] = ChatPromptTemplate.from_template("""
너는 이제 1,2,3단계에서 상담한 내용을 기반으로 자유롭게 대화하는 상담사야.
사용자 반응에 맞게 유연하게 대응하면 될거같아.
이전 대화 : {history}
고객 : {input}
상담사 :
""")

In [71]:
# 라우터 체인 하나 추가 -> 대화 종료 조건(대화를 계속 이어나갈지, 끝낼지 자동으로 결정하는 구문)
# 4단계에 대한 구성
router_prompt = ChatPromptTemplate.from_template("""
대화 : {history}
최근 입력 : {input}

"감사", "도움", "해볼게요", "충분해요", "괜찮아요", "고마워요" 라는 단어가 나오면 -> "종료"
새로운 질문이나 고민이 이어지면 -> "계속"
""")
router_chain = router_prompt | llm | StrOutputParser()

# AI가 단어들의 상황을 보고 대화 종료여부를 자동으로 판단!
def check_if_shoul_end(user_input, history):
  decision = router_chain.invoke({"history": history, "input": user_input}).strip()
  return "종료" in decision # 조건에 만족하니까 True

In [81]:
# # 프롬프트 템플릿을 정의
# prompts = ChatPromptTemplate.from_template("""
# 너는 친절한 상담사야. 고객의 말을 따뜻하게 공감하고 짧은 질문으로 대화를 이어가는 챗봇이야.

# 고객 : {input}
# 상담사 :
# """)

In [82]:
# 상태 및 자동전환 - 전역 상태를 추가하고, 턴에 대한 수행조건을 구현!
# 1. prompt statge
# 2. turn 횟수

state = 1
turn_count = 0

def auto_advance():
  global state # 위에 있는 state를 가져온다.
  if turn_count == 2 and state == 1:
    state = 2
    print("2단계를 분석으로 진행합니다.")
  elif turn_count == 4 and state == 2:
    state = 3
    print("3단계 해결책 제시로 넘어갑니다.")
  elif turn_count == 5 and state == 3:
    state = 4
    print("4단계 자유로운 상담으로 넘어갑니다.")


* summarymemory - 모든 대화를 누적 요약 - 딥러닝 최근 내용만 남고 앞에 있는 내용이 흐려지는 현상 발생
* summarybuffermemory - 최근 대화 일부만 남기고 요약 - 요약 + 일부 원문을 혼합

In [83]:
from re import I
# # 모델 / 프롬프트 / 아웃파서 결합한 LCEL 체인
# chain = prompt | llm | StrOutputParser()

# 1번째 : 모델 / 프롬프트 / 아웃파서 결합한 LCEL 단일 체인
# 2번째 : 단계별 프롬프트 + 메모리 로드 / 저장을 할 수 있는 흐름으로 코드를 대폭 교체
# 3번째 : 4단계 종료 판단 기능 추가 + 마무리 멘트

# 추가 = 종료 기본 플래그 추가
is_ending = False # 종료 판단 플래그

# 2단계
def chat(user_input):
    global turn_count, is_ending # turn_count는 대화가 1턴 끝날때마다 +1 증가

    # 1) 메모리 구현 - memory에 정보가 있으면 가져오고, 없으면 공백으로 가져온다.
    # 첫 턴에서는 당연히 memory가 비어있겠죠? -> 향후에는 고객마다 다른 메모리가 연결될수 있도록 구현
    history = memory.load_memory_variables({}).get('history',"")

    # 추가) 만약에 4단계면 대화 종료 판단
    # 만약 check_if_should_end에 결론이 "종료" in이라는 기능이 구현 True로 변경 -> 대화 종료
    if state == 4:
      is_ending = check_if_should_end(user_input, history)



    # 2) 현재단계 프롬프트를 선택 -> LCEL 실행 -> state는 턴마다 변경이 될 예정!
    chain = prompts[state] | llm | StrOutputParser()
    # 고객의 채팅과 memory에 있는 내용을 기반으로 출력이 가능하도록 만듬
    response = chain.invoke({"input" : user_input, "history" : history})

    # 추가) 종료 멘트를 추가
    # 만약 is_ending = True로 변경이 되었다면
    if is_ending:
      response += "대화가 종료 되었습니다. 언제든 상담을 걸어주세요"

    # 3) 메모리에서 입출력 저장
    # 유저의 질문과 AI답변을 저장
    memory.save_context({"input" : user_input}, {"output" : response})

    # 4) 턴 증가! & 단계 자동 전환
    turn_count += 1
    auto_advance()

    return response

In [84]:
# 한 턴 대화를 실행할 수 있는 기반!
while True:
    # text = input("상담이 필요한 내용을 적어주세요(종료시 quit)을 입력해주세요.").strip()

    # if text.lower() == 'quit':
    #     print("대화를 종료합니다.")
    #     break

    # reply = chain.invoke({"input": text})
    # print(reply)


    # 추가 : 4번 항목이 추가

    label = {1:"공감", 2:"분석", 3:"해결", 4:"자유상담"}[state]
    text  = input(f"{label} 고민이 있으면 얘기해주세요").strip()
    if text.lower() == "quit":
        print("종료합니다.")
        break
    print("상담 내용", chat(text))
    if is_ending:
      print("상담을 마무리 합니다.")
      is_ending = False # 처음부터 다시 대화할 수 있게 유도
      break

공감 고민이 있으면 얘기해주세요짜증난다
상담 내용 상담사: 무슨 일이 있었는지 말씀해 주실 수 있나요?
공감 고민이 있으면 얘기해주세요가족한테 이름을 뺏겼어
2단계를 분석으로 진행합니다.
상담 내용 상담사: 그 상황이 정말 속상하셨겠어요. 이름이 중요한 의미가 있나요?
분석 고민이 있으면 얘기해주세요중요하지
상담 내용 상담사: 이름이 중요한 의미가 있다면, 그 이름이 당신에게 어떤 의미를 지니고 있는지 좀 더 말씀해 주실 수 있을까요? 그리고 가족과의 대화에서 어떤 일이 있었는지 구체적으로 이야기해 주실 수 있나요? 그런 상황이 얼마나 자주 발생하나요?
분석 고민이 있으면 얘기해주세요내가 너무 짓고 싶었던 이름인데
3단계 해결책 제시로 넘어갑니다.
상담 내용 상담사: 그 이름이 당신에게 특별한 의미가 있었다면, 그 이름을 짓고 싶었던 이유가 무엇인지 궁금합니다. 그리고 가족과의 대화에서 어떤 방식으로 이름을 뺏기게 되었는지 구체적으로 말씀해 주실 수 있을까요? 또한, 이런 상황이 얼마나 자주 발생하는지도 알려주시면 좋겠어요.
해결 고민이 있으면 얘기해주세요아니 좀 끝내바
4단계 자유로운 상담으로 넘어갑니다.
상담 내용 ### 대화 요약
대화에서는 한 사용자가 가족으로부터 의미 있는 이름을 뺏기는 상황에 대해 언급하며 감정을 표현했습니다. 그는 그 이름에 대해 특별한 의미가 있었고, 그 이름을 짓고 싶었던 이유를 말하고 싶어 했습니다. 상담사는 상황을 더 깊이 이해하기 위해 질문을 했지만, 사용자는 대화가 길어지자 불만을 표출했습니다.

### 실행 가능한 해결책 3가지
1. **감정 표현의 기회 마련**: 사용자가 자신의 감정을 표현할 수 있는 안전한 공간을 제공하기 위해, 친구나 상담사와의 대화를 통해 자신이 느끼는 감정을 솔직하게 털어놓는 시간을 가지도록 권장합니다.

2. **이름에 대한 의미 재정립**: 사용자가 그 이름의 의미를 스스로 재정립할 수 있도록 도와주는 작업을 제안합니다. 이름이 뺏겼다는 사실을 받아들이고, 그 이름의 대안이나 새로운 아이디

NameError: name 'check_if_should_end' is not defined